# Métodos Numéricos (Bisección y Newton-Raphson)
## Cálculo Multivariable

In [ ]:
import sympy as sp              # Cálculo simbólico
import numpy as np              # Cálculo numérico
import matplotlib.pyplot as plt # Gráficas
from tabulate import tabulate   # Tablas de resultados

# Variable simbólica usada en todo el notebook
x = sp.symbols('x')

# Funciones principales del examen (simbólicas)
f = x**3 - x - 2     # f(x) = x^3 - x - 2
g = sp.exp(-x) - x   # g(x) = e^(-x) - x

print("Funciones definidas:")
sp.pprint(sp.Eq(sp.Function('f')(x), f))
sp.pprint(sp.Eq(sp.Function('g')(x), g))

## Funciones auxiliares (reutilizables)

Se adaptan las funciones de clase `MetodoBiseccion` y `MetodoNewton`. Se les agrega:

- Un parámetro `max_iter` para **detectar la no convergencia** (ciclos o divergencia).
- La opción de recibir una **derivada propia** (útil para funciones por tramos).
- El retorno de la **raíz estimada** y del **número de pasos** para poder comparar métodos.

In [ ]:
def MetodoBiseccion(Funcion, xizq, xder, Precision=1e-6, max_iter=200, mostrar=True):
    """Método de Bisección. Devuelve (raiz, pasos). Imprime tabla con tabulate."""
    Fnum = sp.lambdify(x, Funcion)   # Facilita la evaluación numérica
    Tabla = []
    ErrorAprox = 1
    Pasos = 0
    medio = 0.5 * (xizq + xder)

    while ErrorAprox > Precision and Pasos < max_iter:
        Pasos += 1
        ErrorAprox = 0.5 * (xder - xizq)
        medio = 0.5 * (xizq + xder)          # Punto medio
        Tabla.append([Pasos, xizq, medio, xder,
                      Fnum(xizq), Fnum(medio), Fnum(xder), ErrorAprox])
        # Si el punto medio cae exactamente sobre la raíz, terminamos
        if Fnum(medio) == 0:
            break
        # Actualiza el intervalo
        if Fnum(xizq) * Fnum(medio) < 0:
            xder = medio                     # raíz a la izquierda del medio
        else:
            xizq = medio                     # raíz a la derecha del medio

    if mostrar:
        print('Tabla de Resultados — Método de Bisección')
        print(tabulate(Tabla,
              headers=['Pasos','a (izq)','c (medio)','b (der)','f(a)','f(c)','f(b)','Error Aprox.'],
              numalign='center', tablefmt='fancy_grid',
              floatfmt=['3d','13.10f','13.10f','13.10f','9.2e','9.2e','9.2e','9.3e']))
    return medio, Pasos


def MetodoNewton(Funcion, xest, Precision=1e-6, max_iter=200, Derivada=None, mostrar=True):
    """Método de Newton-Raphson. Devuelve (raiz, pasos, convergio).
    Si Derivada es None se calcula simbólicamente con sympy."""
    if Derivada is None:
        Derivada = sp.diff(Funcion, x)
    Fnum = sp.lambdify(x, Funcion)
    Dnum = sp.lambdify(x, Derivada)

    Tabla = []
    ErrorAprox = 1
    Pasos = 0
    nuevo = xest
    convergio = True

    while ErrorAprox > Precision and Pasos < max_iter:
        Pasos += 1
        try:
            nuevo = xest - Fnum(xest) / Dnum(xest)   # Estimación de la raíz
        except ZeroDivisionError:
            print(f'  Derivada nula en el paso {Pasos}: el método no puede continuar.')
            convergio = False
            break
        ErrorAprox = abs(nuevo - xest)
        Tabla.append([Pasos, xest, nuevo, Fnum(nuevo), ErrorAprox])
        xest = nuevo

    if Pasos >= max_iter and ErrorAprox > Precision:
        convergio = False
        print(f'  ¡No convergió! Se alcanzó el máximo de {max_iter} iteraciones (posible ciclo/divergencia).')

    if mostrar:
        print('Tabla de Resultados — Método de Newton-Raphson')
        print(tabulate(Tabla,
              headers=['Pasos','x estimada','x nuevo','f(x nuevo)','Error Aprox.'],
              numalign='center', tablefmt='fancy_grid',
              floatfmt=['3d','19.16f','19.16f','9.2e','9.2e']))
    return nuevo, Pasos, convergio

## Tema 1 — Bisección y Newton para $f$ y $g$  (20 pts)
$f(x)=x^3-x-2$ en $[1,2]$

In [ ]:
TOL1 = 0.5e-4   # Cuatro decimales correctos

print("===== f(x) = x^3 - x - 2  ·  BISECCIÓN en [1, 2] =====")
raiz_f_bis, pasos_f_bis = MetodoBiseccion(f, 1, 2, Precision=TOL1)
print(f"\nRaíz aproximada (Bisección): {raiz_f_bis:.4f}   en {pasos_f_bis} pasos")

In [ ]:
print("===== f(x) = x^3 - x - 2  ·  NEWTON con x0 = 1.5 =====")
raiz_f_new, pasos_f_new, _ = MetodoNewton(f, 1.5, Precision=TOL1)
print(f"\nRaíz aproximada (Newton): {raiz_f_new:.4f}   en {pasos_f_new} pasos")

### Tema 1 — $g(x)=e^{-x}-x$ en $[0,1]$

In [ ]:
print("===== g(x) = e^(-x) - x  ·  BISECCIÓN en [0, 1] =====")
raiz_g_bis, pasos_g_bis = MetodoBiseccion(g, 0, 1, Precision=TOL1)
print(f"\nRaíz aproximada (Bisección): {raiz_g_bis:.4f}   en {pasos_g_bis} pasos")

In [ ]:
print("===== g(x) = e^(-x) - x  ·  NEWTON con x0 = 0.5 =====")
raiz_g_new, pasos_g_new, _ = MetodoNewton(g, 0.5, Precision=TOL1)
print(f"\nRaíz aproximada (Newton): {raiz_g_new:.4f}   en {pasos_g_new} pasos")

In [ ]:
# Resumen comparativo del Tema 1
resumen1 = [
    ['f(x)', 'Bisección', raiz_f_bis, pasos_f_bis],
    ['f(x)', 'Newton',    raiz_f_new, pasos_f_new],
    ['g(x)', 'Bisección', raiz_g_bis, pasos_g_bis],
    ['g(x)', 'Newton',    raiz_g_new, pasos_g_new],
]
print(tabulate(resumen1,
      headers=['Función','Método','Raíz (4 dec.)','Pasos'],
      numalign='center', tablefmt='fancy_grid',
      floatfmt=['','','.6f','3d']))

### Tema 1 (b) — ¿Qué método es mejor para cada función?

**Para $f(x)=x^3-x-2$:**  el método de **Newton** es claramente superior. Converge en unas pocas iteraciones (3–4) gracias a su **convergencia cuadrática**: el número de decimales correctos aproximadamente se duplica en cada paso. La Bisección, en cambio, necesita alrededor de 15 iteraciones para alcanzar 4 decimales, porque su **convergencia es lineal** (el error se reduce a la mitad en cada paso, ganando solo $\log_{10}2\approx 0.3$ decimales por iteración). Como $f$ es un polinomio suave con derivada bien comportada cerca de la raíz, Newton no tiene riesgo de fallar aquí.

**Para $g(x)=e^{-x}-x$:**  nuevamente **Newton** es el más eficiente (converge en 3–4 pasos). La función es monótona decreciente y su derivada $g'(x)=-e^{-x}-1$ nunca se anula, por lo que Newton es muy estable. La Bisección también funciona perfectamente porque hay cambio de signo garantizado en $[0,1]$, pero requiere muchas más iteraciones.

**Conclusión.** Newton gana en velocidad para ambas funciones (menos iteraciones por su orden cuadrático). La ventaja de la Bisección es su **robustez**: siempre converge si hay cambio de signo, sin necesidad de derivada ni de un buen punto inicial. En resumen: Newton por rapidez, Bisección por seguridad.

## Tema 2 — Newton para $h(x)=x^3-2x+2$  (25 pts)

$$h(x) = x^3 - 2x + 2, \qquad h'(x) = 3x^2 - 2$$

In [ ]:
h = x**3 - 2*x + 2          # h(x)
dh = sp.diff(h, x)          # h'(x) = 3x^2 - 2
print("h(x)  ="); sp.pprint(h)
print("h'(x) ="); sp.pprint(dh)

### Tema 2 (a) — $x_0 = 0$  (Newton entra en un ciclo)

Con $x_0=0$:  $h(0)=2$, $h'(0)=-2 \Rightarrow x_1 = 0 - \tfrac{2}{-2} = 1$.
Luego $h(1)=1$, $h'(1)=1 \Rightarrow x_2 = 1 - \tfrac{1}{1} = 0$.

In [ ]:
print("===== h(x)  ·  NEWTON con x0 = 0  (se espera un CICLO) =====")
raiz_h_a, pasos_h_a, conv_a = MetodoNewton(h, 0, Precision=1e-12, max_iter=50)
print(f"\n¿Convergió? {conv_a}  (las iteraciones oscilan entre 0 y 1)")

### Tema 2 (b) — $x_0 = 0.7$  (región "mala")

Aquí $h'(0.7) = 3(0.49)-2 = -0.53$, un valor **pequeño y cercano al punto de inflexión** $x=\sqrt{2/3}\approx 0.816$ donde $h'=0$. Dividir entre una derivada pequeña produce un salto enorme, lanzando la iteración muy lejos. Se observa el comportamiento errático antes de (eventualmente) caer hacia la raíz real o agotar `max_iter`.

In [ ]:
print("===== h(x)  ·  NEWTON con x0 = 0.7  (región cercana a la inflexión) =====")
raiz_h_b, pasos_h_b, conv_b = MetodoNewton(h, 0.7, Precision=1e-12, max_iter=50)
print(f"\n¿Convergió? {conv_b}  ·  Pasos: {pasos_h_b}  ·  Estimación final: {raiz_h_b:.12f}")

### Tema 2 (c) — $x_0 = -1.5$  (converge a la raíz real)

La única raíz real de $h$ está cerca de $x\approx -1.769292354239$. Con $x_0=-1.5$ se parte **cerca** de ella y con derivada $h'(-1.5)=3(2.25)-2=4.75$ grande y bien condicionada. Se usa $\text{TOL}=10^{-12}$.

In [ ]:
print("===== h(x)  ·  NEWTON con x0 = -1.5  (TOL = 1e-12) =====")
raiz_h_c, pasos_h_c, conv_c = MetodoNewton(h, -1.5, Precision=1e-12, max_iter=50)
print(f"\n¿Convergió? {conv_c}  ·  Pasos: {pasos_h_c}")
print(f"Raíz real (12 decimales): {raiz_h_c:.12f}")
print(f"Valor de referencia      : -1.769292354239")

### Tema 2 (d) — Explicación del comportamiento

- **$x_0 = 0$ (ciclo).** El punto inicial cae en una zona donde las tangentes rebotan entre $0$ y $1$ formando un **ciclo de período 2**. Newton no diverge ni converge: queda atrapado. Influye que cerca de esta región la derivada es pequeña (el punto de inflexión está en $x\approx 0.816$, donde $h'=0$). La guardia `max_iter` es esencial para detener el programa.

- **$x_0 = 0.7$ (región mala).** Está muy cerca del punto de inflexión donde $h'\approx 0$. La tangente es casi horizontal, así que su corte con el eje $x$ se va lejísimos. El método se comporta de forma errática (puede divergir o saltar a otra cuenca). Es una zona donde **no conviene iniciar** Newton.

- **$x_0 = -1.5$ (converge).** Está **cerca de la raíz real** y la derivada ahí es grande y de signo estable. Esto cumple las condiciones para la convergencia cuadrática de Newton, y se obtienen los 12 decimales en muy pocas iteraciones.

**Importancia de elegir bien $x_0$.** Newton es muy rápido pero **solo converge localmente**: garantiza convergencia si se parte suficientemente cerca de la raíz y la derivada no se anula. Un mal $x_0$ puede producir ciclos, divergencia o convergencia a una raíz distinta. Conviene graficar la función o usar Bisección para obtener un buen punto de partida.

## Tema 3 — Polinomio $f(x)=x^4-10x^3+33x^2-39x+10$  (30 pts)

$$f(x) = x^4 - 10x^3 + 33x^2 - 39x + 10, \qquad \text{TOL}=10^{-8}$$

Tiene 4 raíces reales. Se localizan gráficamente, se aproximan con Bisección y Newton, y se comparan con las raíces exactas.

In [ ]:
p = x**4 - 10*x**3 + 33*x**2 - 39*x + 10   # polinomio del Tema 3
TOL3 = 1e-8
pn = sp.lambdify(x, p)   # versión numérica

# Verificación de signos en los enteros
print("Evaluación en enteros (cambios de signo):")
for xi in range(0, 6):
    print(f"  f({xi}) = {pn(xi):6.2f}")

### Tema 3 (a) — Gráfica y localización de raíces

Por los cambios de signo: raíz en $[0,1]$, raíz exacta en $x=2$, raíz en $[3,4]$ y raíz en $[4,5]$.

In [ ]:
xv = np.linspace(-0.5, 6, 600)
yv = pn(xv)

plt.figure(figsize=(9,5))
plt.plot(xv, yv, 'b-', label='f(x)')
plt.axhline(0, color='k', linewidth=1)          # eje x
plt.axvline(0, color='gray', linewidth=0.5)
# Marcar visualmente los intervalos de las raíces
for (a_i, b_i) in [(0,1), (1,3), (3,4), (4,5)]:
    plt.axvspan(a_i, b_i, alpha=0.08, color='red')
plt.title('Tema 3 — f(x) = x⁴ - 10x³ + 33x² - 39x + 10')
plt.xlabel('x'); plt.ylabel('f(x)')
plt.ylim([-15, 20]); plt.xlim([-0.5, 6])
plt.grid(); plt.legend(loc='lower right')
plt.show()

### Tema 3 (b) — Bisección para las dos primeras raíces

Raíz 1 en $[0,1]$ y raíz 2 ($x=2$) detectada desde $[1.5,\,2.5]$. Tolerancia $10^{-8}$.

In [ ]:
print("===== Raíz 1 · BISECCIÓN en [0, 1] =====")
r1_bis, n1_bis = MetodoBiseccion(p, 0, 1, Precision=TOL3)
print(f"\nRaíz 1 (Bisección): {r1_bis:.8f}   en {n1_bis} pasos")

In [ ]:
print("===== Raíz 2 · BISECCIÓN en [1.5, 2.5]  (raíz exacta x = 2) =====")
r2_bis, n2_bis = MetodoBiseccion(p, 1.5, 2.5, Precision=TOL3)
print(f"\nRaíz 2 (Bisección): {r2_bis:.8f}   en {n2_bis} pasos")

### Tema 3 (c) — Newton para las dos últimas raíces

Raíz 3 desde $x_0=3.5$ (en $[3,4]$) y raíz 4 desde $x_0=4.5$ (en $[4,5]$).

In [ ]:
print("===== Raíz 3 · NEWTON con x0 = 3.5 =====")
r3_new, n3_new, _ = MetodoNewton(p, 3.5, Precision=TOL3)
print(f"\nRaíz 3 (Newton): {r3_new:.8f}   en {n3_new} pasos")

In [ ]:
print("===== Raíz 4 · NEWTON con x0 = 4.5 =====")
r4_new, n4_new, _ = MetodoNewton(p, 4.5, Precision=TOL3)
print(f"\nRaíz 4 (Newton): {r4_new:.8f}   en {n4_new} pasos")

### Tema 3 (d) — Raíces exactas y comparación de errores

Se obtienen las raíces exactas con `sp.real_roots` y se calculan el error absoluto y el error relativo porcentual.

In [ ]:
# Raíces exactas (simbólicas -> numéricas, ordenadas)
raices_exactas = sorted([complex(r).real for r in sp.real_roots(p)])
print("Raíces exactas:", [f"{r:.10f}" for r in raices_exactas])

# Asociar cada aproximación con su raíz exacta más cercana
aprox = [
    ('Bisección', r1_bis, n1_bis),
    ('Bisección', r2_bis, n2_bis),
    ('Newton',    r3_new, n3_new),
    ('Newton',    r4_new, n4_new),
]

tabla3 = []
for i, (metodo, ap, pasos) in enumerate(aprox, start=1):
    exacta = min(raices_exactas, key=lambda r: abs(r - ap))   # exacta más cercana
    err_abs = abs(ap - exacta)
    err_rel = err_abs / abs(exacta) * 100 if exacta != 0 else 0.0
    tabla3.append([f'Raíz {i}', metodo, ap, exacta, err_abs, err_rel, pasos])

print(tabulate(tabla3,
      headers=['Raíz','Método','Aproximada','Exacta','Error abs.','Error rel. %','Pasos'],
      numalign='center', tablefmt='fancy_grid',
      floatfmt=['','','13.10f','13.10f','9.2e','9.2e','3d']))

## Tema 4 — Función por tramos $H(x)$  (25 pts)

$$H(x)=\begin{cases} x^3 - 2x^2 + 3 & x \le 2\\[4pt] x^4 - 4x^3 - x^2 + 23 & 2 \le x < 4\\[4pt] -329 + 25x^2 + x^3 - \dfrac{x^5}{8} & x \ge 4 \end{cases}$$

Derivadas por tramo:

$$H'(x)=\begin{cases} 3x^2 - 4x & x \le 2\\[4pt] 4x^3 - 12x^2 - 2x & 2 \le x < 4\\[4pt] 50x + 3x^2 - \dfrac{5x^4}{8} & x \ge 4 \end{cases}$$

Nota: $H(-1)=0$, por lo que $x=-1$ es una raíz exacta.

In [ ]:
def H(x):
    """Función por tramos H(x). Acepta escalar o arreglo de numpy."""
    x = np.asarray(x, dtype=float)
    y = np.empty_like(x)
    y = np.where(x <= 2,
                 x**3 - 2*x**2 + 3,
                 np.where(x < 4,
                          x**4 - 4*x**3 - x**2 + 23,
                          -329 + 25*x**2 + x**3 - x**5/8))
    return y if y.shape else float(y)

def dH(x):
    """Derivada por tramos H'(x)."""
    x = np.asarray(x, dtype=float)
    y = np.where(x <= 2,
                 3*x**2 - 4*x,
                 np.where(x < 4,
                          4*x**3 - 12*x**2 - 2*x,
                          50*x + 3*x**2 - 5*x**4/8))
    return y if y.shape else float(y)

# Verificación de valores clave
print("Valores de H en puntos clave:")
for xi in [-1, 0, 2, 3, 4]:
    print(f"  H({xi:2d}) = {H(xi):8.3f}")

### Tema 4 (a) — Gráfica de $H(x)$ en $[-2,6]$

In [ ]:
xv = np.linspace(-2, 6, 800)
yv = H(xv)

plt.figure(figsize=(9,5))
plt.plot(xv, yv, 'b-', label='H(x)')
plt.axhline(0, color='k', linewidth=1)        # eje x
plt.axvline(2, color='gray', linestyle='--', linewidth=0.7)  # frontera de tramo
plt.axvline(4, color='gray', linestyle='--', linewidth=0.7)  # frontera de tramo
plt.title('Tema 4 — Función por tramos H(x)')
plt.xlabel('x'); plt.ylabel('H(x)')
plt.ylim([-30, 30]); plt.xlim([-2, 6])
plt.grid(); plt.legend(loc='upper right')
plt.show()

Por la gráfica y los valores: una raíz exacta en $x=-1$ (cuenca $[-2,-0.5]$) y otra raíz en $[2,3]$ porque $H(2)=3>0$ y $H(3)=-13<0$.

Como $H$ y $H'$ son funciones de numpy por tramos (no expresiones de SymPy), se definen **versiones específicas** de Bisección y Newton que reciben funciones de Python directamente.

In [ ]:
def BiseccionFunc(Func, xizq, xder, Precision=1e-6, max_iter=200, mostrar=True):
    """Bisección que recibe una función de Python (no símbolo de SymPy)."""
    Tabla = []; ErrorAprox = 1; Pasos = 0
    medio = 0.5 * (xizq + xder)
    while ErrorAprox > Precision and Pasos < max_iter:
        Pasos += 1
        ErrorAprox = 0.5 * (xder - xizq)
        medio = 0.5 * (xizq + xder)
        Tabla.append([Pasos, xizq, medio, xder,
                      Func(xizq), Func(medio), Func(xder), ErrorAprox])
        if Func(medio) == 0:          # punto medio justo sobre la raíz
            break
        if Func(xizq) * Func(medio) < 0:
            xder = medio
        else:
            xizq = medio
    if mostrar:
        print('Tabla de Resultados — Método de Bisección')
        print(tabulate(Tabla,
              headers=['Pasos','a (izq)','c (medio)','b (der)','H(a)','H(c)','H(b)','Error Aprox.'],
              numalign='center', tablefmt='fancy_grid',
              floatfmt=['3d','13.10f','13.10f','13.10f','9.2e','9.2e','9.2e','9.3e']))
    return medio, Pasos


def NewtonFunc(Func, Deriv, xest, Precision=1e-6, max_iter=200, mostrar=True):
    """Newton que recibe función y derivada de Python (no símbolos)."""
    Tabla = []; ErrorAprox = 1; Pasos = 0; nuevo = xest; convergio = True
    while ErrorAprox > Precision and Pasos < max_iter:
        Pasos += 1
        d = Deriv(xest)
        if d == 0:
            print(f'  Derivada nula en el paso {Pasos}.'); convergio = False; break
        nuevo = xest - Func(xest) / d
        ErrorAprox = abs(nuevo - xest)
        Tabla.append([Pasos, xest, nuevo, Func(nuevo), ErrorAprox])
        xest = nuevo
    if Pasos >= max_iter and ErrorAprox > Precision:
        convergio = False
    if mostrar:
        print('Tabla de Resultados — Método de Newton-Raphson')
        print(tabulate(Tabla,
              headers=['Pasos','x estimada','x nuevo','H(x nuevo)','Error Aprox.'],
              numalign='center', tablefmt='fancy_grid',
              floatfmt=['3d','19.16f','19.16f','9.2e','9.2e']))
    return nuevo, Pasos, convergio

### Tema 4 (b) — Newton para las dos primeras raíces

Raíz 1 con $x_0=-1.5$ y raíz 2 con $x_0=2.5$.

In [ ]:
print("===== Raíz 1 · NEWTON con x0 = -1.5 =====")
H_r1_new, H_n1_new, _ = NewtonFunc(H, dH, -1.5, Precision=1e-8)
print(f"\nRaíz 1 (Newton): {H_r1_new:.8f}   en {H_n1_new} pasos")

In [ ]:
print("===== Raíz 2 · NEWTON con x0 = 2.5 =====")
H_r2_new, H_n2_new, _ = NewtonFunc(H, dH, 2.5, Precision=1e-8)
print(f"\nRaíz 2 (Newton): {H_r2_new:.8f}   en {H_n2_new} pasos")

### Tema 4 (c) — Bisección para las dos primeras raíces

Raíz 1 en $[-2,\,0]$ (contiene a $x=-1$) y raíz 2 en $[2,\,3]$ ($H(2)=3>0$, $H(3)=-13<0$).

In [ ]:
print("===== Raíz 1 · BISECCIÓN en [-2, 0]  (raíz exacta x = -1) =====")
H_r1_bis, H_n1_bis = BiseccionFunc(H, -2, 0, Precision=1e-8)
print(f"\nRaíz 1 (Bisección): {H_r1_bis:.8f}   en {H_n1_bis} pasos")

In [ ]:
print("===== Raíz 2 · BISECCIÓN en [2, 3] =====")
H_r2_bis, H_n2_bis = BiseccionFunc(H, 2, 3, Precision=1e-8)
print(f"\nRaíz 2 (Bisección): {H_r2_bis:.8f}   en {H_n2_bis} pasos")

In [ ]:
# Resumen comparativo del Tema 4
resumen4 = [
    ['Raíz 1', 'Newton',    H_r1_new, H_n1_new],
    ['Raíz 1', 'Bisección', H_r1_bis, H_n1_bis],
    ['Raíz 2', 'Newton',    H_r2_new, H_n2_new],
    ['Raíz 2', 'Bisección', H_r2_bis, H_n2_bis],
]
print(tabulate(resumen4,
      headers=['Raíz','Método','Aproximada','Pasos'],
      numalign='center', tablefmt='fancy_grid',
      floatfmt=['','','13.10f','3d']))
print("\nNota: la Raíz 1 es exacta en x = -1; Newton y Bisección lo confirman.")